# ⚠️ Setup Instructions

To run this notebook successfully, please ensure you complete the following steps:

1. **Switch to T4 GPU**: Ensure your runtime is configured to use a T4 GPU. In the menu, go to **Runtime > Change runtime type**, select **T4 GPU** under *Hardware accelerator*, and click **Save**.
2. **Shared Folder Shortcut**: Make sure you have access to the shared repository Drive folder and have created a **shortcut** from it directly to your own Google Drive's home directory (`My Drive`). This allows the code to resolve the path `/content/drive/MyDrive/traffic-risk-agent-shared/scene-risk-traffic-safety-analysis` correctly.
3. **Drive Permissions**: When you select **Runtime > Run all**, Google Colab will prompt you for Google Drive access permissions. You must authorize full access to your Drive for the file reading and writing processes to run without interruption.

# Traffic Risk Agent Pipeline
This notebook walks you through setting up and running the Traffic Risk Agent Pipeline using YOLO spatial metadata extraction and an Ollama-hosted vision-language model (Moondream) to assess traffic scene risk.

## Cell 1: Team Instructions & Drive Mount
We will mount Google Drive, check GPU status, and configure the working directory to access our shared repository.

In [ ]:
import os
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

# Check GPU availability
!nvidia-smi

# Set working directory to project repository
REPO_PATH = '/content/drive/MyDrive/traffic-risk-agent-shared/scene-risk-traffic-safety-analysis'
os.chdir(REPO_PATH)
print(f"Current Working Directory: {os.getcwd()}")

# List directory contents
print("Directory contents:")
print(os.listdir('.'))

Mounted at /content/drive
Mon Oct 19 18:05:36 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   37C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+---------------------

In [ ]:
# ── Utility: Generate / Reset Ground Truth Table ───────────────────────
# Uncomment the line below ONLY when you want to scan data/raw_images and
# pre-populate / reset data/ground_truth.csv with fresh assignments:

# !python src/utils/build_ground_truth.py


## Cell 2: Install & Start Ollama Background Daemon
We will install Ollama, launch it as a background service, wait for initialization, and pull the `moondream` vision model.

In [ ]:
import os
import subprocess
import time
import urllib.request

# 1. Install/Update hardware utilities and latest Ollama binary
!sudo apt-get update && sudo apt-get install -y zstd pciutils lshw
!curl -fsSL https://ollama.com/install.sh | sh

# 2. Kill stale Ollama daemon (exact process name only, preventing subshell kills)
!pkill -9 -x ollama || true
time.sleep(2)

# 3. Check installed Ollama version (must be >= 0.4.0)
!/usr/local/bin/ollama --version

# 4. Set CUDA & concurrency environment variables
env = os.environ.copy()
env["CUDA_VISIBLE_DEVICES"] = "0"
env["LD_LIBRARY_PATH"] = "/usr/lib64-nvidia:/usr/local/cuda/lib64:" + env.get("LD_LIBRARY_PATH", "")
env["OLLAMA_NUM_PARALLEL"] = "1"
env["OLLAMA_MAX_LOADED_MODELS"] = "1"
env["OLLAMA_FLASH_ATTENTION"] = "1"
env["OLLAMA_KEEP_ALIVE"] = "10m"

# 5. Start a FRESH Ollama daemon in the background with GPU support
print("Starting fresh Ollama daemon with GPU support...")
process = subprocess.Popen(
    ["/usr/local/bin/ollama", "serve"],
    env=env,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

# 6. Wait for the server to bind to port 11434
print("Waiting for Ollama server to become available...")
for i in range(15):
    try:
        urllib.request.urlopen("http://localhost:11434")
        print("Ollama server is active and running successfully with GPU support!")
        break
    except Exception:
        time.sleep(2)
else:
    print("⚠️ Ollama daemon failed to start. Check daemon logs.")

# 7. Pull primary model (gemma3:12b)
print("Pulling model...")
# !/usr/local/bin/ollama pull gemma3:12b

# To pull Qwen VL for the follow-up comparison, uncomment:
# !/usr/local/bin/ollama pull qwen2.5vl:7b
!/usr/local/bin/ollama pull qwen3-vl:8b-instruct

!/usr/local/bin/ollama list


Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ InRelease
Hit:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64  InRelease
Hit:4 http://security.ubuntu.com/ubuntu noble-security InRelease
Hit:5 http://archive.ubuntu.com/ubuntu noble InRelease
Hit:6 https://r2u.stat.illinois.edu/ubuntu noble InRelease
Hit:7 http://archive.ubuntu.com/ubuntu noble-updates InRelease
Hit:8 http://archive.ubuntu.com/ubuntu noble-backports InRelease
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble InRelease
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu noble InRelease
Fetched 3,917 B in 4s (1,005 B/s)
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependen

## Cell 3: Install Python Dependencies
We will install the required Python libraries using standard Colab pip commands.

In [ ]:
# Fallback for uv package manager:
# !pip install uv && !uv sync

!pip install ultralytics opencv-python pandas scipy scikit-learn ollama

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.9/46.9 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 81.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.6/97.6 kB 11.8 MB/s eta 0:00:00


## Cell 5: Full Pipeline Execution
Now we will run the main evaluation pipeline. Results are written directly to `output/experiment_results.csv` on your persistent Drive storage.

In [ ]:
GT_URL = "https://docs.google.com/spreadsheets/d/13Ldjb5q13H5s9ip-RqOCp8QufUqQE-g6ltsinGnGnkc/edit?usp=sharing"

# !python main.py --provider gemini --gt-url "$GT_URL"


# Run the 10-image smoke test:
# !python main.py --provider ollama --model gemma3:8b --limit 10
!python main.py --provider ollama --model qwen3-vl:8b-instruct --gt-url "$GT_URL"
# !python main.py --provider ollama --model qwen2.5vl:7b --gt-url "$GT_URL"

# Once verified, run the full 58 images by removing --limit 10:
# !python main.py --provider ollama --model gemma3:12b


Processing 64 image(s) from /content/drive/.shortcut-targets-by-id/1mk9x4sJXp9iQy755-z_I7gjJxLpgunSx/traffic-risk-agent-shared/scene-risk-traffic-safety-analysis/data/raw_images

⬇️  Downloading ground truth from Google Sheets …
   ✓ Downloaded 64 rows.

Loaded 64 ground-truth entries from https://docs.google.com/spreadsheets/d/13Ldjb5q13H5s9ip-RqOCp8QufUqQE-g6ltsinGnGnkc/edit?usp=sharing.

Initialising YOLO perception extractor …
Initialising VLM client (ollama) …

[1/64] Processing: 0.png
  YOLO: 4 road user(s) detected
  VLM Condition A (image-only) …
    → Attention Score: 39/100 [Stay Alert] (130.98s)
      • traffic_density       :  2/10 | The street is relatively empty with a few pedestrians and cyclists.
      • vulnerable_road_users :  4/10 | Pedestrians and cyclists are present, requiring vigilance.
      • conflict_proximity    :  3/10 | No immediate conflict, but pedestrians are close to the vehicle path.
      • road_complexity       :  5/10 | The road has tram tracks, add

## Cell 6: Results Visualization & Evaluation Metrics
Finally, we will load the pipeline run results, compare predictions with ground truth, and compute evaluation metrics (Pearson, Spearman, and Classification scores).

In [ ]:
import pandas as pd
from scipy.stats import pearsonr, spearmanr
from sklearn.metrics import precision_score, recall_score, f1_score

results_path = 'output/experiment_results.csv'
try:
    df = pd.read_csv(results_path)
    print("Preview of experiment results:")
    display(df.head())

    if 'ground_truth' in df.columns and 'predicted_risk' in df.columns:
        # Continuous Evaluation Metrics
        r_coef, _ = pearsonr(df['predicted_risk'], df['ground_truth'])
        rho_coef, _ = spearmanr(df['predicted_risk'], df['ground_truth'])
        print(f"\nPearson Correlation (r): {r_coef:.3f}")
        print(f"Spearman Rank Correlation (rho): {rho_coef:.3f}")

        # Classification Evaluation Metrics (Threshold >= 70)
        threshold = 70
        y_true = (df['ground_truth'] >= threshold).astype(int)
        y_pred = (df['predicted_risk'] >= threshold).astype(int)

        precision = precision_score(y_true, y_pred, zero_division=0)
        recall = recall_score(y_true, y_pred, zero_division=0)
        f1 = f1_score(y_true, y_pred, zero_division=0)

        print(f"\nClassification metrics (Threshold >= {threshold}):")
        print(f"Precision: {precision:.3f}")
        print(f"Recall:    {recall:.3f}")
        print(f"F1 Score:  {f1:.3f}")
    else:
        print("Columns 'ground_truth' or 'predicted_risk' missing from the results csv.")
except FileNotFoundError:
    print(f"Results file not found at {results_path}. Ensure the pipeline completed successfully.")

Preview of experiment results:


,image_id,yolo_detections,condition_a_score,condition_a_state,condition_a_time_sec,condition_b_score,condition_b_state,condition_b_time_sec,total_time_sec,ground_truth_score,cond_a_traffic_density_score,cond_b_traffic_density_score,cond_a_vulnerable_road_users_score,cond_b_vulnerable_road_users_score,cond_a_conflict_proximity_score,cond_b_conflict_proximity_score,cond_a_road_complexity_score,cond_b_road_complexity_score,cond_a_impaired_visibility_score,cond_b_impaired_visibility_score
0,0,4,NaN,Unknown,0.0,NaN,Unknown,0.0,0.0,19,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1,6,NaN,Unknown,0.0,NaN,Unknown,0.0,0.0,28,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,10,7,NaN,Unknown,0.0,NaN,Unknown,0.0,0.0,30,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,100,0,NaN,Unknown,0.0,NaN,Unknown,0.0,0.0,14,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,102,4,NaN,Unknown,0.0,NaN,Unknown,0.0,0.0,35,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Columns 'ground_truth' or 'predicted_risk' missing from the results csv.
